## 1. Configuración e Importaciones
En esta celda importamos las librerías y definimos las constantes del proyecto (nombres de datasets, algoritmos, etc.).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from scipy import stats
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, roc_auc_score
import pathlib as pl

# CONFIGURACIÓN DE RUTAS ---
# AJUSTAR estas rutas según estructura de nuestras carpetas
TYPES = ['original', 'estandarizado', 'normalizado']
VARIANTS = ['', '_PCA95', '_PCA80']
FOLDER_PREFIX = 'conj'
VALIDATING_FILE_REGEX = 'validating*.csv'
DATA_PATH = './kfolds_data'
MODEL_PATH = './trained_models'
MODEL_EXT = 'joblib'                       
PATH_PREDICCIONES = "./predicciones"                                   # Ruta donde se guardarán las predicciones  
PATH_METRICAS = "./metricas"                                           # Ruta donde se guardarán las métricas                

# Crear carpetas si no existen
os.makedirs(PATH_PREDICCIONES, exist_ok=True)
os.makedirs(PATH_METRICAS, exist_ok=True)

# Lista con los algoritmos a evaluar y número de folds
MODELOS = ["KNN", "SVM", "NaiveBayes", "RandomForest"]

## 2. Funciones Auxiliares (Carga y Guardado)
Necesitamos funciones para cargar los datos y modelos correspondientes a cada iteración y almacenar los resultados.

In [ ]:
# Funciones de carga, evaluación y guardado de resultados
def cargar_datos_test(filename):
    try:
        df = pd.read_csv(filename)
        X = df.drop('species', axis=1)
        Y = df['species']
        return X, Y
    except FileNotFoundError:
        print(f"Falta archivo {filename}")
        return None, None

def cargar_modelo(filename):
    try:
        return joblib.load(filename)
    except FileNotFoundError:
        print(f"Falta modelo {filename}")
        return None

def calcular_metricas(y_true, y_pred, y_proba):
    """
    Calcula las métricas solicitadas.
    Adapta fórmulas binarias a multiclase usando macro-average.
    """
    # Métricas básicas de sklearn (usando 'macro' para multiclase)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')      # F1-Score
    rec = recall_score(y_true, y_pred, average='macro') # Sensibilidad
    prec = precision_score(y_true, y_pred, average='macro') # Precisión
    
    # Métricas derivadas de la Matriz de Confusión (Especificidad, FNR, FPR)
    cm = confusion_matrix(y_true, y_pred)
    
    # Cálculo de TP, TN, FP, FN por clase
    FP = cm.sum(axis=0) - np.diag(cm)  
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    # Evitar división por cero
    epsilon = 1e-7 
    
    # Promedios macro de las tasas
    spec_macro = np.mean(TN / (FP + TN + epsilon))
    fnr_macro = np.mean(FN / (TP + FN + epsilon))
    fpr_macro = np.mean(FP / (FP + TN + epsilon))
    
    # AUC 
    auc_val = 0
    if y_proba is not None:
        try:
            auc_val = roc_auc_score(y_true, y_proba, multi_class='ovr')
        except:
            pass

    return {
        "Exactitud": acc, "F1": f1, "Sensibilidad": rec, "Recall": rec,
        "Precision": prec, "Especificidad": spec_macro,
        "FNR": fnr_macro, "FPR": fpr_macro, "AUC": auc_val
    }

def guardar_resultados(dataset, fold, metodo, y_true, y_pred, y_proba):
    """
    Guarda DOS archivos: 
     Predicciones crudas (para ensembles y futuros análisis) y métricas calculadas (para la tabla de resultados finales)
    """
    # Guardar Predicciones (CSV Grande)
    data_pred = {"y_true": y_true, "y_pred": y_pred}
    if y_proba is not None:
        for i in range(y_proba.shape[1]):
            data_pred[f"prob_{i}"] = y_proba[:, i]
    
    os.makedirs(f"{PATH_PREDICCIONES}/{metodo}/{dataset}", exist_ok=True)
    pd.DataFrame(data_pred).to_csv(f"{PATH_PREDICCIONES}/{metodo}/{dataset}/{metodo}{fold}_{dataset}_predicts.csv", index=False)

    # Guardar Métricas (CSV Pequeño con una fila) 
    metricas = calcular_metricas(y_true, y_pred, y_proba)
    metricas['Dataset'] = dataset
    metricas['Fold'] = fold
    metricas['Method'] = metodo
    
    # Reordenar para que las columnas identificadoras vayan primero
    cols = ['Dataset', 'Fold', 'Method'] + [k for k in metricas.keys() if k not in ['Dataset', 'Fold', 'Method']]
    
    os.makedirs(f"{PATH_METRICAS}/{metodo}/{dataset}", exist_ok=True)
    pd.DataFrame([metricas], columns=cols).to_csv(f"{PATH_METRICAS}/{metodo}/{dataset}/{metodo}{fold}_{dataset}_metrics.csv", index=False)

## 3. Creación de predicciones y estadisticas de modelos base y ensembles
En esta sección iteramos sobre cada dataset y fold para generar las predicciones y métricas de los modelos base (KNN, SVM, NB, RF) utilizando los datos de test. Además, cuando los cuatro modelos se ejecutan correctamente, calculamos y evaluamos tres métodos de ensemble (Votación, Media y Mediana) combinando sus resultados para intentar mejorar el rendimiento final.

In [ ]:
for tipo in TYPES:
    for var in VARIANTS:
        dataset_name = f"{tipo}{var}"
        validating_route = pl.Path(f'{DATA_PATH}/{FOLDER_PREFIX}_{dataset_name}/')
        validating_csv = sorted([x.name for x in validating_route.glob(VALIDATING_FILE_REGEX)])

        for fold, csv_file in enumerate(validating_csv):
            # Cargar datos de test
            X_test, Y_test = cargar_datos_test(validating_route / validating_csv[fold])
            if X_test is None or Y_test is None:
                continue
            
            # Listas para guardar las predicciones de los 4 modelos base para este fold actual
            ensemble_preds_clases = []
            ensemble_preds_probabilidades = []
            modelos_validos_count = 0
            class_labels = None

            for model in MODELOS:
                model_folder = pl.Path(f'{MODEL_PATH}/{model}/{dataset_name}')
                validating_models = sorted([x.name for x in model_folder.glob(f'{model}*.{MODEL_EXT}')])

                # Cargar modelo entrenado
                modelo = cargar_modelo(model_folder / validating_models[fold])
                if modelo is None:
                    continue
                    
                # Guardamos las clases del modelo (necesario para los ensemble)
                if class_labels is None and hasattr(modelo, "classes_"):
                    class_labels = modelo.classes_

                # Realizar predicciones individuales
                Y_pred = modelo.predict(X_test)
                try:
                    Y_proba = modelo.predict_proba(X_test)
                except:
                    Y_proba = None
                    
                # Guardar resultados
                guardar_resultados(dataset_name, fold + 1, model, Y_test, Y_pred, Y_proba)

                # Añadir a las listas para el Ensemble
                ensemble_preds_clases.append(Y_pred)
                ensemble_preds_probabilidades.append(Y_proba)
                modelos_validos_count += 1

            if modelos_validos_count == 4:
                # A) ENSEMBLE VOTACIÓN (Moda de las clases)
                moda_resultado = stats.mode(np.stack(ensemble_preds_clases), axis=0, keepdims=True)
                y_pred_votacion = moda_resultado.mode[0]
                y_proba_votacion = np.mean(np.stack(ensemble_preds_probabilidades), axis=0)
                guardar_resultados(dataset_name, fold + 1, "Ensemble_Votacion", Y_test, y_pred_votacion, y_proba_votacion)

                # B) ENSEMBLE MEDIA (Promedio de probabilidades)
                y_proba_media = np.mean(np.stack(ensemble_preds_probabilidades), axis=0)
                indices_media = np.argmax(y_proba_media, axis=1) # Devuelve 0, 1, 2
                y_pred_media = class_labels[indices_media]       # Traducimos a etiqueta real
                guardar_resultados(dataset_name, fold + 1, "Ensemble_Media", Y_test, y_pred_media, y_proba_media)

                # C) ENSEMBLE MEDIANA (Mediana de probabilidades)
                y_proba_mediana = np.median(np.stack(ensemble_preds_probabilidades), axis=0)
                indices_mediana = np.argmax(y_proba_mediana, axis=1) # Devuelve 0, 1, 2
                y_pred_mediana = class_labels[indices_mediana]       # Traducimos a etiqueta real
                guardar_resultados(dataset_name, fold + 1, "Ensemble_Mediana", Y_test, y_pred_mediana, y_proba_mediana)

            else:
                print(f"\nSolo {modelos_validos_count} modelos validos en fold {fold}. No se ha aplicado Ensemble.")

# Hacemos la media de las metricas

In [ ]:
for model in MODELOS:
    for tipo in TYPES:
        for var in VARIANTS:
            metrics_path = pl.Path(f"{PATH_METRICAS}/{model}/{tipo}{var}/")
            all_metrics_files = sorted([x.name for x in metrics_path.glob('*.csv')])
            df_all_metrics = pd.concat([pd.read_csv(metrics_path / f) for f in all_metrics_files], ignore_index=True)
            # print(df_all_metrics)
            df_mean_metrics = df_all_metrics.drop('Fold', axis=1).mean(numeric_only=True)
            df_mean_metrics['Method'] = model
            df_mean_metrics['Dataset'] = f"{tipo}{var}"
            print(f"\nMétricas medias para {model} en {tipo}{var}:\n{df_mean_metrics}\n")
            df_mean_metrics.to_csv(f"{PATH_METRICAS}/{model}/{tipo}{var}/{model}_{tipo}{var}_average_metrics.csv", index=False)